In [1]:
import os
import sys
from pyspark.sql import SparkSession

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    '--conf spark.driver.extraClassPath="C:/data/spark/jars/iceberg-spark-runtime-4.0_2.13-1.10.0.jar" '
    "pyspark-shell"
)

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable


BUCKET_NAME = "iceberg"
RPT_WAREHOUSE_PATH = f"s3a://{BUCKET_NAME}/iceberg/WideWorldImportersDW"
WH_CATALOG_NAME = "reporting"

spark = SparkSession.builder \
    .appName("Iceberg Reporting Access") \
    .config("spark.jars.packages", 
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2,"
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.warehouse", RPT_WAREHOUSE_PATH) \
    .config("spark.hadoop.fs.s3a.endpoint", "http://127.0.0.1:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .getOrCreate()

In [3]:
df_purchase = spark.table("reporting.Fact.Purchase").alias("Purchase")
df_stock_holding = spark.table("reporting.Fact.Stock_Holding").alias("Stock_Holding")
df_order = spark.table("reporting.Fact.Order").alias("Order")
df_movement = spark.table("reporting.Fact.Movement").alias("Movement")
df_sale = spark.table("reporting.Fact.Sale").alias("Sale")
df_payment_method = spark.table("reporting.Dimension.payment_method").alias("payment_method")
df_supplier = spark.table("reporting.Dimension.supplier").alias("supplier")
df_city = spark.table("reporting.Dimension.city").alias("city")
df_stock_item = spark.table("reporting.Dimension.stock_item").alias("stock_item")
df_customer = spark.table("reporting.Dimension.customer").alias("customer")
df_date = spark.table("reporting.Dimension.dates").alias("dates")
df_transaction_type = spark.table("reporting.Dimension.transaction_type").alias("transaction_type")
df_employee = spark.table("reporting.Dimension.employee").alias("employee")
df_transaction = spark.table("reporting.Fact.Transaction").alias("Transaction")

In [4]:
df_city.show(5, truncate=False)

+--------+-----------+------------+--------------+-------------+-------------+---------------+--------+----------------+-------------------------------------------------------------------+--------------------------+-------------------+--------------------------+-----------+
|City_Key|WWI_City_ID|City        |State_Province|Country      |Continent    |Sales_Territory|Region  |Subregion       |Location                                                           |Latest_Recorded_Population|Valid_From         |Valid_To                  |Lineage_Key|
+--------+-----------+------------+--------------+-------------+-------------+---------------+--------+----------------+-------------------------------------------------------------------+--------------------------+-------------------+--------------------------+-----------+
|1456    |9660       |Eads        |Colorado      |United States|North America|Rocky Mountain |Americas|Northern America|[E6 10 00 00 01 0C BD 1B 0B 0A 83 3D 43 40 2A 5D B0 0A 